# GeoSR-4 — EDSR with heteroscedastic uncertainty (Kaggle GPU)

Phase 6 (PRD section 38-39), never actually trained for real until now -- see `decisions.md` D016/D018. Single-pass uncertainty head (mean + log-variance, Gaussian NLL loss) instead of a 5x Monte Carlo ensemble, for compute reasons.

**Before running, in the notebook's right-hand Settings panel:**
- Accelerator: GPU (T4 x2 or P100)
- Internet: ON (needed for git clone + dataset download)

Kaggle gives ~30 GPU-hours/week on the free tier -- this run (~20 epochs) takes a fraction of that.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
Kaggle's default image already has PyTorch with CUDA -- only the packages it's missing get installed.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Train
Same EDSR-baseline config (16 blocks, 64 channels) as Phase 3, but out_channels=8 (mean + log-variance) and Gaussian NLL loss instead of L1. Gradient clipping is on by default (D016 -- this loss can diverge violently at the wrong LR; confirmed stable at 1e-4).

In [ ]:
!python ml/training/train_edsr_uncertainty.py \
  --epochs 20 \
  --batch-size 16 \
  --n-blocks 16 \
  --n-channels 64 \
  --lr 1e-4 \
  --checkpoint-dir experiments/edsr_uncertainty \
  --log-every 20

## 4. Full validation-set evaluation (real numbers, not the 50-sample per-epoch estimate)
Reports PSNR/SSIM/SAM/ERGAS on the mean prediction, plus the calibration number -- correlation between predicted uncertainty and actual error. Positive and >0.3-0.4 means the uncertainty map is actually informative, not just noise.

In [ ]:
!python ml/evaluation/evaluate_checkpoint.py \
  --checkpoint experiments/edsr_uncertainty/edsr_unc_epoch19.pt \
  --model-type edsr --uncertainty --n-blocks 16 --n-channels 64

## 5. Get the checkpoint out
Kaggle has no `google.colab.files.download` -- easiest path: zip it and it'll appear in the notebook's **Output** tab (right-hand panel) after you save a version, downloadable from there. Or use the Kaggle API (`kaggle kernels output`) from your own machine.

In [ ]:
import shutil
shutil.copy("experiments/edsr_uncertainty/edsr_unc_epoch19.pt", "/kaggle/working/edsr_unc_epoch19.pt")
print("copied to /kaggle/working/ -- visible in the Output tab once you save a version of this notebook")